# RAG over academic papers — step-by-step walkthrough

This notebook walks through each stage of the pipeline with its intermediate
output, then runs three questions that demonstrate correct grounded answers
and honest refusals. It imports the real pipeline from `src/` — nothing is
re-implemented here.

Run from the project root, after `python -m scripts.build_embeddings`.

In [1]:
import os
os.chdir("..")            # move from notebooks/ up to project root

In [2]:
from dotenv import load_dotenv
load_dotenv()   # OPENAI_API_KEY into the environment before any OpenAI client

True

## Stage 1 — Extraction and chunking

PDFs are layout, not text. We extract text, repair line-wrap artifacts, strip
references and figure-axis noise, then split into ~400-token chunks by packing
whole paragraphs (with overlap so boundary-straddling facts survive).

In [3]:
from src.extract import extract_text
from src.chunk import chunk_text, count_tokens

sample_pdf = "data/papers/2013_Znamenskiy_and_Zador.pdf"  # adjust if needed
text = extract_text(sample_pdf)
chunks = chunk_text(text)

print(f"Extracted {len(text)} characters -> {len(chunks)} chunks")
print(f"Chunk token counts: {[count_tokens(c) for c in chunks][:8]} ...")
print("\n--- first chunk ---\n")
print(chunks[0][:500])

Extracted 44599 characters -> 28 chunks
Chunk token counts: [413, 407, 407, 405, 411, 409, 402, 405] ...

--- first chunk ---

Corticostriatal neurons in auditory cortex drive

decisions during auditory discrimination

Petr Znamenskiy1,2 & Anthony M. Zador2

The neural pathways by which information about the acoustic

world reaches the auditory cortex are well characterized, but how

auditory representations are transformed into motor commands is

not known. Here we use a perceptual decision-making task in rats

to study this transformation. We demonstrate the role of corticostriatal projection neurons in auditory decis


## Stage 2 — Embeddings (loaded, not recomputed)

Each chunk is a 1536-dim vector from `text-embedding-3-small`, where similar
meaning = nearby vectors. We embedded once and persisted; here we load. The
vectors are unit-length (norm ~ 1), which is what makes dot product equal
cosine similarity at retrieval time.

In [4]:
from src.embed import load_corpus
import numpy as np

vectors, corpus_chunks = load_corpus()
print(f"Corpus: {vectors.shape[0]} vectors x {vectors.shape[1]} dims")
print(f"Example vector norm: {np.linalg.norm(vectors[0]):.4f}  (unit length)")

Corpus: 491 vectors x 1536 dims
Example vector norm: 1.0001  (unit length)


## Stage 3 — Vector store

FAISS `IndexFlatIP` holds the vectors and searches them exactly (Flat) by
inner product (IP) = cosine similarity for unit vectors.

In [5]:
from src.store import build_index

index = build_index(vectors)
print(f"Index built over {index.ntotal} vectors")

Index built over 491 vectors


## Stage 4 — Retrieval

A query is embedded the same way, then we find the k nearest chunk-vectors
and map their positions back to text. Below: the top chunks for a known
question — read whether they are actually on-topic.

In [6]:
from src.retrieve import retrieve

retrieved = retrieve(
    "What behavioural task did the rats perform in the cloud-of-tones experiment?",
    index, corpus_chunks, k=5,
)
for i, r in enumerate(retrieved):
    print(f"--- result {i+1} ---")
    print(r[:300], "\n")

--- result 1 ---
We used tetrode recordings to characterize the activity of individual

neurons in the auditory cortex and the auditory striatum while rats

performed the task. Auditory striatal neurons have been characterized

previously in anaesthetized and passively listening animals14,15 but not

during behaviou 

--- result 2 ---
After reaching a weight of 200to 250g, rats were placed on a water deprivation

schedule and behavioural training commenced. The rats were placed in a soundproof behavioural chamber and presented with three choice ports. The rats were

trained to first poke into the centre port, wait for the onset o 

--- result 3 ---
Figure 1 | Cloud-of-tones task. a, Format of the behavioural task. b, Example

stimulus spectrograms at 2100, 0 and 1100 tones pers. c, Psychometric curve

from a single rat (error bars, 95% confidence interval).

Macmillan Publishers Limited. All rights reserved

compared performance on stimulated  

--- result 4 ---
display stable performa

## Stage 5 — Generation: the three test questions

The full pipeline (retrieve -> augment prompt -> generate). Three questions:

1. **In corpus** — should give a correct, grounded answer.
2. **Clearly out of corpus** (capital of France) — should refuse, even though
   the model knows the answer from training.
3. **Plausibly in-domain but absent** (dopamine / reward prediction error) —
   the harder refusal: retrieval returns related-but-wrong chunks, and the
   system should still say it cannot find the answer rather than confabulate.

In [7]:
from src.generate import rag

questions = [
    "What behavioural task did the rats perform in the cloud-of-tones experiment?",
    "What is the capital of France?",
    "What is the role of dopamine in reward prediction error?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag(q, index, corpus_chunks)}\n")

Q: What behavioural task did the rats perform in the cloud-of-tones experiment?
A: The rats performed a discrimination task where they were trained to poke into a center port, wait for the onset of an auditory stimulus, and then select one of two other ports to receive a water reward based on the auditory stimulus presented.

Q: What is the capital of France?
A: I cannot find it in the provided papers.

Q: What is the role of dopamine in reward prediction error?
A: I cannot find it in the provided papers.



**What the three answers demonstrate:** the system answers correctly and
grounded when the corpus contains the answer, and refuses honestly when it
does not — including when retrieval returns semantically-related but non-
answering chunks. That grounding-plus-refusal behaviour is what separates a
trustworthy RAG system from one that confabulates.